# 03 · ChemKAN from scratch — a step-by-step neural ODE for chemical kinetics

This notebook builds the **entire ChemKAN model inline**, one small piece at a time, and
trains it as a neural ODE on the isothermal **biodiesel** dataset. Everything is
self-contained and inspectable so you can stop at any cell and poke at shapes, values,
and gradients.

We build bottom-up, mirroring the package modules:

| step | piece | package module |
|:--:|:--|:--|
| 1 | Gaussian RBF primitive | `kan/_common.py` |
| 2 | `RBFEdgeFunctions` (the einsum) | `kan/rbf.py` |
| 3 | `AddKANLayer`, `LeanKANLayer` | `kan/layers.py` |
| 4 | `KineticCore` → verify **156** params | `model.py` |
| 5 | min-max input scaling (the tanh-saturation fix) | `normalization.py` |
| 6 | `KineticDynamics` (the ODE right-hand side) | `dynamics.py` |
| 7 | integrate + train (neural ODE, direct autograd) | `solver.py`, `training.py` |
| 8 | evaluate & plot; then how hydrogen adds thermo + PINN | `model.py` |

> Biodiesel is isothermal, so we only integrate the **species** `Y` with an externally
> supplied temperature — the simplest full training loop. The temperature-rate
> superstructure (`dT/dt`) and the PINN term are introduced conceptually in Step 8.


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torchdiffeq import odeint

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False)
DEVICE = torch.device("cpu")

: 

## Step 0 · Load and inspect the data

The repo's generator stores states as `(cases, times, vars)`. Training code wants
`(T, B, vars)`, so we permute. Biodiesel has **6 species**, each trajectory has its own
constant temperature, and the archive ships **train-only** `u_min`/`u_max` for the
Eq. 18 normalized loss.


In [ ]:
data = np.load("../data/generated/biodiesel.npz", allow_pickle=True)
species = [str(s) for s in data["species"]]
m = len(species)                                                   # 6 species

def to_TBx(a):  # (cases, T, vars) -> (T, B, vars)
    return torch.as_tensor(a, dtype=torch.float32).permute(1, 0, 2).contiguous()

states = to_TBx(data["train_states"]).to(DEVICE)                  # (T, B, m)
t = torch.as_tensor(data["t"], dtype=torch.float32, device=DEVICE)  # (T,)
T_const = torch.as_tensor(data["train_T"], dtype=torch.float32, device=DEVICE)  # (B,)
Y0 = states[0]                                                     # (B, m) initial states
u_min = torch.as_tensor(data["u_min"], dtype=torch.float32)       # (m,) species, train-only
u_max = torch.as_tensor(data["u_max"], dtype=torch.float32)

T_len, B, _ = states.shape
print(f"species  = {species}")
print(f"states (T,B,m) = {tuple(states.shape)},  t = {tuple(t.shape)}")
print(f"batch B  = {B} trajectories,   temperatures range "
      f"{T_const.min():.1f}..{T_const.max():.1f} K")

: 

## Paper notation before implementation: biodiesel is $7\to4\to6$

The notebook below implements the biodiesel kinetic core from ChemKAN Eq. 16:

$$
\mathrm{KAN}_{\mathrm{kin}}
=
\left(\Psi^{\mathrm{lean}}_1\circ\Psi^{\mathrm{add}}_0\right)(\mathbf u).
$$

With six biodiesel species,

$$
\mathbf{x}_0=\mathbf u=
[\mathrm{TG},\mathrm{ROH},\mathrm{DG},\mathrm{MG},\mathrm{GL},\mathrm{R'CO_2R},T]^\top
\in\mathbb R^7,
$$

the paper's reported four-node hidden layer gives

$$
\boxed{
\mathbf{x}_0\in\mathbb R^7
\xrightarrow{\Psi^{\mathrm{add}}_0}
\mathbf{x}_1\in\mathbb R^4
\xrightarrow{\Psi^{\mathrm{lean}}_1,\;n^{\mathrm{mu}}=2}
\mathbf{x}_2=\dot{\tilde{\mathbf u}}\in\mathbb R^6
}.
$$

For a generic layer $l$,

$$
\phi_{l,\alpha,\beta}
$$

means the learned edge function from source node $\beta$ in layer $l$ to destination node $\alpha$ in layer $l+1$.

Thus:

- $\Phi_0$ has 4 rows and 7 columns;
- $\Phi_1$ has 6 rows and 4 columns;
- rows are destination nodes;
- columns are source nodes.

In code, $\alpha\to o$, $\beta\to i$, and the RBF/grid index in Eq. 11 becomes `k`.


## Step 1 · The Gaussian RBF primitive (Eq. 12)

One shared function so the math lives in exactly one place. A trailing basis axis is
appended, so it serves inputs of any rank.


In [ ]:
def gaussian(x, centers, h):
    # psi_k(x) = exp(-(x-c_k)^2 / (2 h^2));  appends a trailing basis axis (..., K)
    r = x.unsqueeze(-1) - centers
    return torch.exp(-r ** 2 / (2 * h ** 2))

# quick smoke: a bump centered at 0 peaks at x=0
c = torch.linspace(-1, 1, 5); hh = 0.5
print("gaussian(0):", gaussian(torch.tensor(0.0), c, hh))

## Step 2 · `RBFEdgeFunctions` — the vectorized edge grid

`tanh` normalizes inputs onto the center grid, then the `"bik,oik->boi"` einsum
evaluates every edge $\phi_{o,i}(x_i)=\sum_k w_{o,i,k}\psi_k(x_i)$ at once
(see notebook `02` for the einsum walkthrough). `use_base_act=False` matches the
paper's parameter counts — no `w_base` is created.


### Paper-to-code identity for `RBFEdgeFunctions`

For a fixed layer $l$, one paper edge is

$$
\phi_{l,\alpha,\beta}(x_{l,\beta})
=
\sum_{k=1}^{N}
w^\psi_{l,\alpha,\beta,k}\psi_k(x_{l,\beta})
+
w^b_{l,\alpha,\beta}b(x_{l,\beta}).
$$

The RBF path in `RBFEdgeFunctions.forward` evaluates all such edges in parallel:

$$
E^{(l)}_{b,\alpha,\beta}
=
\sum_k
\psi_{b,\beta,k}w^\psi_{l,\alpha,\beta,k}.
$$

That is exactly

```python
torch.einsum("bik,oik->boi", psi, w_rbf)
```

where `o = alpha`, `i = beta`, and `k = basis/grid index`.

The sum over `k` produces **one scalar per edge**.  
The AddKAN or LeanKAN class then combines those edge scalars over the source-node index $\beta$ to produce a node value.


In [ ]:
class RBFEdgeFunctions(nn.Module):
    def __init__(self, in_features, out_features, num_basis, grid=(-1.0, 1.0),
                 *, use_base_act):
        super().__init__()
        if num_basis < 2:
            raise ValueError("num_basis must be >= 2")
        centers = torch.linspace(grid[0], grid[1], num_basis)
        self.h = (grid[1] - grid[0]) / (num_basis - 1)
        self.register_buffer("centers", centers)                 # fixed, not trained
        self.w_rbf = nn.Parameter(torch.randn(out_features, in_features, num_basis) * 0.1)
        if use_base_act:
            self.base = nn.SiLU()
            self.w_base = nn.Parameter(torch.zeros(out_features, in_features))
        else:
            self.base = None
            self.register_parameter("w_base", None)              # not counted, not trained

    def forward(self, x):                                         # (B, in) -> (B, out, in)
        x = torch.tanh(x)                                        # KAN internal normalization
        psi = gaussian(x, self.centers, self.h)                 # (B, in, K)
        edge = torch.einsum("bik,oik->boi", psi, self.w_rbf)    # (B, out, in)
        if self.w_base is not None:
            edge = edge + torch.einsum("bi,oi->boi", self.base(x), self.w_base)
        return edge

e = RBFEdgeFunctions(7, 4, 3, use_base_act=False)
print("edge shape:", tuple(e(torch.randn(5, 7)).shape))          # (5, 4, 7)

## Step 3 · `AddKANLayer` and `LeanKANLayer`

They share the edge grid and differ only in how they collapse the input axis:
`AddKAN` sums; `LeanKAN` multiplies the first `n_mu` inputs and adds the rest
(with the `n_mu=0` empty-product guard).


In [ ]:
class AddKANLayer(nn.Module):
    # y_i = sum_j phi_{i,j}(x_j)   (ChemKAN Eq. 7)
    def __init__(self, in_features, out_features, num_basis, grid=(-1.0, 1.0),
                 *, use_base_act):
        super().__init__()
        self.edges = RBFEdgeFunctions(in_features, out_features, num_basis, grid,
                                      use_base_act=use_base_act)
    def forward(self, x):
        return self.edges(x).sum(dim=-1)                         # sum over inputs


class LeanKANLayer(nn.Module):
    # product of the first n_mu inputs + sum of the rest (LeanKAN Eq. 8-10)
    def __init__(self, in_features, out_features, n_mu, num_basis, grid=(-1.0, 1.0),
                 *, use_base_act):
        super().__init__()
        if not 0 <= n_mu <= in_features:
            raise ValueError("need 0 <= n_mu <= in_features")
        self.n_mu = n_mu
        self.edges = RBFEdgeFunctions(in_features, out_features, num_basis, grid,
                                      use_base_act=use_base_act)
    def forward(self, x):
        g = self.edges(x)                                        # (B, out, in)
        add = g[..., self.n_mu:].sum(dim=-1)
        if self.n_mu == 0:
            return add                                           # empty-product guard
        mult = g[..., :self.n_mu].prod(dim=-1)
        return mult + add

print("Add :", tuple(AddKANLayer(7, 4, 3, use_base_act=False)(torch.randn(5, 7)).shape))
print("Lean:", tuple(LeanKANLayer(4, 6, n_mu=2, num_basis=3,
                                  use_base_act=False)(torch.randn(5, 4)).shape))

## Step 4 · `KineticCore` and the 156-parameter check

The species-rate network is $\mathrm{KAN_{kin}}=\Psi^{\text{lean}}_1\circ\Psi^{\text{add}}_0$:
an AddKAN mapping $[Y,T]\in\mathbb{R}^{m+1}$ to `hidden_dim`, then a LeanKAN mapping
`hidden_dim` to $m$. With the paper's biodiesel hyperparameters
(`hidden_dim=4, num_basis=3, n_mu=2`, base **off**) it must have exactly **156**
trainable parameters:

$$(\underbrace{7\cdot4}_{\text{add edges}}+\underbrace{4\cdot6}_{\text{lean edges}})\cdot\underbrace{3}_{K}=156.$$


### Exact meaning of the two layer objects

**First layer (`self.add`)**

$$
l=0,\quad n_0=7,\quad n_1=4.
$$

Its function matrix is $\Phi_0\in\mathcal F^{4\times7}$.  
After each of the 28 edge functions has been evaluated,

$$
x_{1,\alpha}
=
\sum_{\beta=1}^{7}
\phi_{0,\alpha,\beta}(x_{0,\beta}),
\qquad \alpha=1,\ldots,4.
$$

This is what `AddKANLayer` implements.

**Second layer (`self.lean`)**

$$
l=1,\quad n_1=4,\quad n_2=6,\quad n^{\mathrm{mu}}=2.
$$

Its function matrix is $\Phi_1\in\mathcal F^{6\times4}$.  
For every species-rate output $\alpha$,

$$
x_{2,\alpha}
=
\underbrace{\phi_{1,\alpha,1}(x_{1,1})
            \phi_{1,\alpha,2}(x_{1,2})}_{\text{multiplicative sublayer}}
+
\underbrace{\phi_{1,\alpha,3}(x_{1,3})
            +\phi_{1,\alpha,4}(x_{1,4})}_{\text{additive sublayer}}.
$$

Those six $x_{2,\alpha}$ values are the six components of
$\dot{\tilde{\mathbf u}}$ in ChemKAN Eq. 13.


In [ ]:
class KineticCore(nn.Module):
    # u=[Y,T] (B, m+1) -> dY/dt (B, m)
    def __init__(self, species_dim, hidden_dim, num_basis, n_mu, *, use_base_act):
        super().__init__()
        self.add = AddKANLayer(species_dim + 1, hidden_dim, num_basis,
                               use_base_act=use_base_act)
        self.lean = LeanKANLayer(hidden_dim, species_dim, n_mu=n_mu, num_basis=num_basis,
                                 use_base_act=use_base_act)
    def forward(self, u):
        return self.lean(self.add(u))

HIDDEN, NUM_BASIS, N_MU = 4, 3, 2                                 # paper biodiesel config
core = KineticCore(m, HIDDEN, NUM_BASIS, N_MU, use_base_act=False).to(DEVICE)

n_params = sum(p.numel() for p in core.parameters() if p.requires_grad)
print(f"KineticCore trainable params = {n_params}")
assert n_params == 156, "should reproduce the paper's biodiesel count"
print("matches paper biodiesel count (156)  ✓")

## Step 5 · Min-max input scaling — the tanh-saturation fix

The KAN's internal `tanh` saturates on raw Kelvin (`tanh(330 K) = 1`), erasing all
temperature dependence. We fix this by min-max scaling the **complete physical state**
`[Y, T]` to a sane range *before* the KAN, using **train-only** statistics. Crucially
this scales only the *copy handed to the model* — the ODE state and the returned
derivatives stay physical.

The archive stores species-only stats `(m,)`, so we append T's train range to get a
full-state `(m+1,)` normalizer. It is **not clipped** (held-out values may exit `[0,1]`).


In [ ]:
class MinMaxNormalizer(nn.Module):
    def __init__(self, u_min, u_max, eps=1e-12):
        super().__init__()
        self.register_buffer("u_min", u_min.float())
        self.register_buffer("range", (u_max - u_min).clamp_min(eps).float())
    def normalize(self, x):
        return (x - self.u_min) / self.range

# full-state (m+1) normalizer: species stats from archive + temperature's train range
u_min_full = torch.cat([u_min, T_const.min().reshape(1)])
u_max_full = torch.cat([u_max, T_const.max().reshape(1)])
input_normalizer = MinMaxNormalizer(u_min_full, u_max_full).to(DEVICE)

print("full-state u_min:", input_normalizer.u_min)
# witness: raw path collapses 320 K vs 340 K; scaled path separates them
raw = torch.tensor([320.0, 340.0])
print("tanh(raw T)      :", torch.tanh(raw), " <- saturated, identical")
print("tanh(scaled T)   :", torch.tanh(input_normalizer.normalize(
    torch.cat([torch.zeros(2, m), raw[:, None]], -1))[:, -1]), " <- separated")

## Step 6 · `KineticDynamics` — the ODE right-hand side

The solver calls `f(t, Y)`. Biodiesel temperature is constant per trajectory, so we
supply it externally, assemble the **physical** `u=[Y,T]`, scale the *copy* for the KAN,
and return the **physical** `dY/dt`. State in, state out — both physical.


In [ ]:
class ConstantTemperature(nn.Module):
    def __init__(self, temperature):                             # (B,) or (B,1)
        super().__init__()
        temperature = torch.as_tensor(temperature, dtype=torch.float32)
        if temperature.ndim == 1:
            temperature = temperature.unsqueeze(-1)
        self.register_buffer("temperature", temperature)
    def forward(self, t):
        return self.temperature                                  # (B, 1), t ignored


class KineticDynamics(nn.Module):
    def __init__(self, kinetic_core, temperature, *, input_normalizer):
        super().__init__()
        self.kinetic = kinetic_core
        self.temperature = temperature
        self.input_normalizer = input_normalizer
    def forward(self, t, Y):                                     # (B, m) -> (B, m)
        T = self.temperature(t)                                 # (B, 1) physical Kelvin
        u_physical = torch.cat([Y, T], dim=-1)                  # (B, m+1) physical
        u_model = (u_physical if self.input_normalizer is None
                   else self.input_normalizer.normalize(u_physical))
        return self.kinetic(u_model)                            # physical dY/dt

temp = ConstantTemperature(T_const).to(DEVICE)
dynamics = KineticDynamics(core, temp, input_normalizer=input_normalizer).to(DEVICE)
print("dY/dt at t0:", tuple(dynamics(t[0], Y0).shape))          # (B, m)

## Step 7 · Integrate and train (neural ODE, direct autograd)

We integrate `dY/dt` over the full time grid with `torchdiffeq.odeint` (`dopri5`), and
backprop **through the solver** (direct autograd — *not* `odeint_adjoint`, *not* the
paper's Forward Sensitivity Analysis). The loss is the Eq. 18 trajectory MSE on
**normalized** species (species-only subset of the normalizer).

> For a quick interactive run we use a few hundred epochs; the paper trains for ~10⁴.
> Bump `EPOCHS` for a closer fit.


In [ ]:
species_norm = MinMaxNormalizer(u_min, u_max).to(DEVICE)        # species-only, for the loss
target_norm = species_norm.normalize(states)                    # (T, B, m) normalized truth

def trajectory_mse(pred):
    pred_norm = species_norm.normalize(pred)
    per_state = ((pred_norm - target_norm) ** 2).mean(dim=-1)   # (1/m) sum_k -> (T, B)
    return per_state.sum(dim=0).mean()                          # sum time, mean batch

EPOCHS, LR = 400, 2e-3                                           # demo; paper uses ~1e4
opt = torch.optim.Adam(core.parameters(), lr=LR)
history = []
for epoch in range(EPOCHS):
    opt.zero_grad()
    pred = odeint(dynamics, Y0, t, method="dopri5", rtol=1e-6, atol=1e-8)  # (T, B, m)
    loss = trajectory_mse(pred)
    loss.backward()
    opt.step()
    history.append(loss.item())
    if epoch % 50 == 0:
        print(f"epoch {epoch:4d}  loss {loss.item():.3e}")
print(f"final loss {history[-1]:.3e}")

In [ ]:
plt.figure(figsize=(5, 3))
plt.semilogy(history)
plt.xlabel("epoch"); plt.ylabel("normalized MSE"); plt.title("training loss")
plt.tight_layout(); plt.show()

## Step 8 · Evaluate — predicted vs. true trajectories

In [ ]:
with torch.no_grad():
    pred = odeint(dynamics, Y0, t, method="dopri5", rtol=1e-6, atol=1e-8)

traj = 0                                                         # inspect one trajectory
fig, ax = plt.subplots(figsize=(6, 4))
tn = t.cpu().numpy()
for k, name in enumerate(species):
    ax.plot(tn, states[:, traj, k].cpu(), "o", ms=3, alpha=0.5,
            color=f"C{k}")
    ax.plot(tn, pred[:, traj, k].cpu(), "-", color=f"C{k}", label=name)
ax.set_xlabel("time"); ax.set_ylabel("mass fraction")
ax.set_title(f"biodiesel trajectory {traj}: dots = truth, lines = ChemKAN")
ax.legend(fontsize=8, ncol=2); plt.tight_layout(); plt.show()

## Step 9 · From biodiesel to hydrogen — thermo + PINN

Biodiesel is isothermal, so we integrated species only. The **full ChemKAN** also
predicts the temperature rate and integrates the coupled `[Y, T]` state
(ChemKAN Eq. 14–17):

$$\frac{dT}{dt}=\underbrace{\text{Linear}_{m\to1}\!\big(\tfrac{dY}{dt}\big)}_{\text{Eq. 14, }\approx-h_i/c_p}
\;+\;\underbrace{\mathrm{KAN_{cor}}(u)}_{\text{Eq. 17, AddKAN }(m{+}1)\to1}$$

The rest of the recipe is identical — same edge einsum, same layers, same neural-ODE
training — plus two hydrogen-specific pieces:

- **`ChemKANDynamics`** integrates the full `(B, m+1)` state instead of species-only.
- a **PINN term** penalizes elemental-mass drift (physical species, `alpha_PINN=1e-4`).

Here is the full model forward, for reference (this is `model.ChemKAN`):


In [ ]:
class ThermodynamicSuperstructure(nn.Module):
    # dT/dt = Linear(dY/dt) + KAN_cor(u)   (Eq. 14-15, 17)
    def __init__(self, species_dim, num_basis, *, use_base_act):
        super().__init__()
        self.linear = nn.Linear(species_dim, 1, bias=False)     # m coefficients, Eq. 14
        self.correction = AddKANLayer(species_dim + 1, 1, num_basis,
                                      use_base_act=use_base_act)  # KAN_cor, Eq. 17
    def forward(self, u, dYdt):
        return self.linear(dYdt) + self.correction(u)           # (B, 1)


class ChemKAN(nn.Module):
    # u=[Y,T] (B, m+1) -> [dY/dt, dT/dt] (B, m+1)
    def __init__(self, species_dim, hidden_dim, num_basis, n_mu, *, use_base_act):
        super().__init__()
        self.kinetic = KineticCore(species_dim, hidden_dim, num_basis, n_mu,
                                   use_base_act=use_base_act)
        self.thermo = ThermodynamicSuperstructure(species_dim, num_basis,
                                                   use_base_act=use_base_act)
    def forward(self, u):
        dYdt = self.kinetic(u)
        dTdt = self.thermo(u, dYdt)
        return torch.cat([dYdt, dTdt], dim=-1)

# hydrogen: 9 species (N2 included) -> full state is m+1 = 10 ([Y_1..Y_9, T]); param check
hm = 9
hyd = ChemKAN(hm, hidden_dim=3, num_basis=5, n_mu=3, use_base_act=False)
n = sum(p.numel() for p in hyd.parameters() if p.requires_grad)
print(f"hydrogen ChemKAN params = {n}")
assert n == 344, "reproduces the paper's hydrogen count"
print("matches paper hydrogen count (344)  ✓")

## Recap

You built the full ChemKAN from the ground up and trained it as a neural ODE:

1. **Gaussian primitive** → 2. **`RBFEdgeFunctions`** (the `"bik,oik->boi"` einsum) →
3. **Add/Lean layers** → 4. **`KineticCore`** (156 params ✓) → 5. **min-max input
scaling** (the tanh-saturation fix, physical state preserved) → 6. **`KineticDynamics`**
(physical ODE RHS) → 7. **neural-ODE training** via `odeint` + direct autograd →
8. **evaluation** → 9. the **thermo + PINN** extension for hydrogen (344 params ✓).

Every cell is inspectable, so this notebook doubles as the debugging surface for the
`chemkan/src/` package: paste a layer here, print shapes, check gradients, then port
the fix back.
